In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import string
# pip install nltk
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('wordnet')
import gensim
from gensim.utils import simple_preprocess
# pip install spacy
import spacy
nlp = spacy.load("en_core_web_sm")
# pip install presidio-anonymizer
# from presidio_anonymizer import AnonymizerEngine

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
comment_df = pd.read_csv('Forum comment ferpa(in).csv', usecols = ['id', 'course_id', 'body'])
thread_df = pd.read_csv('Forum thread ferpa(in).csv', usecols = ['id', 'course_id', 'body'])
# json_df = pd.read_json('anonymized_ed discussion_discussion_data.json', usecols = ['text'])

In [4]:
combined_df = pd.concat([comment_df, thread_df], ignore_index=True)
# pd.merge(comment_df, thread_df, on=['id', 'course_id', 'body'], how='inner')
print(combined_df.head())
combined_df.to_csv('combined_forum.csv', index=False)

                                                  id  \
0  \x35633165393062633336313031333039623030303363...   
1  \x35626565353734343336313031333039626330303137...   
2  \x35626164353931333362306162633039626530303364...   
3  \x35623765323431393362306162633039396430303162...   
4  \x35623762626238383362306162633039393130303138...   

                        course_id  \
0  course-v1:GTx+CS1301xII+1T2018   
1  course-v1:GTx+CS1301xII+1T2018   
2  course-v1:GTx+CS1301xII+1T2018   
3  course-v1:GTx+CS1301xII+1T2018   
4  course-v1:GTx+CS1301xII+1T2018   

                                                body  
0  Enter your answer without spaces! I was going ...  
1                              SmartBook that is :)   
2  What is the answer for this question? I tried ...  
3  Passed with output printed: 12 On the bubble  ...  
4  I had the same problem with this question, I w...  


In [5]:
# extracted_columns = combined_df[['id', 'course_id', 'body']]
# print(extracted_columns.head())
# print(extracted_columns.info())
# print(extracted_columns.describe())

In [6]:
combined_df['body'] = combined_df['body'].str.replace('PERSON', '', regex=False)
combined_df['body'] = combined_df['body'].str.lower()
print(combined_df['body'].head())
# cleaned_comments.str.lower()
# print(cleaned_comments.head(20))
combined_df[combined_df['body'].str.count('smartbook') > 0]

0    enter your answer without spaces! i was going ...
1                                smartbook that is :) 
2    what is the answer for this question? i tried ...
3    passed with output printed: 12 on the bubble  ...
4    i had the same problem with this question, i w...
Name: body, dtype: object


,id,course_id,body
1,\x35626565353734343336313031333039626330303137...,course-v1:GTx+CS1301xII+1T2018,smartbook that is :)
82,\x35633630373564323365623965363039623830303232...,course-v1:GTx+CS1301xI+1T2019,there is a link on one of the pages are you go...
223,\x35626434613062613336313031333039386630303064...,course-v1:GTx+CS1301xI+3T2018,"course:cs1301xiinformation:courseguide,faqs,an..."
265,\x35623434663639393131336239303038346330303132...,course-v1:GTx+CS1301xI+1T2018,oh that's a good point -- that's actually stil...
691,\x35386336626464306631373035363037393230303161...,course-v1:GTx+CS1301x+1T2017,i can't open chapter 1.1 of the smartbook eith...
...,...,...,...
17218,\x35383965336139386461643636633038363230303034...,course-v1:GTx+CS1301x+1T2017,"in the smartbook practice questions, i keep fi..."
18303,\x35393162626137653232613866623037396630303032...,course-v1:GTx+CS1301x+1T2017,i am submitting this post to help the professo...
18331,\x35386563333332656631373035363037633830303262...,course-v1:GTx+CS1301x+1T2017,smartbook is not accessable
18344,\x35386464373732333962653431643038646430303237...,course-v1:GTx+CS1301x+1T2017,"if you take a a4 paper, i see only half vertic..."


In [7]:
def remove_punctuation(text):
    for char in text:
        if char in string.punctuation:
            text = text.replace(char, '')
    return text

In [8]:
combined_df['body'] = combined_df['body'].apply(lambda x:remove_punctuation(x))
print(combined_df['body'].head(10))

0    enter your answer without spaces i was going c...
1                                  smartbook that is  
2    what is the answer for this question i tried e...
3    passed with output printed 12 on the bubble  t...
4    i had the same problem with this question i wi...
5    ah i see and for   i do it the other way aroun...
6    i knew in my bones it was just a typo  i just ...
7    i upgraded to  and the exam questions i had an...
8                                im interested as well
9    hi no i have not could you please provide the ...
Name: body, dtype: object


In [9]:
stopwords_list = stopwords.words('english')
stopwords_list.extend(['ok','okay','see', 'ah', 'way', 'hi', 'hello', 'hey', 'please', 'thank', 'thanks', 'would', 'like', 'know', 'can', 'could', 'get', 'got', 'go', 'going', 'want', 'wanna', 'wont', 'will', 'wouldnt', 'should', 'shouldnt', 'im'])


In [10]:
def stopword_clean(text):
    word_list=[]
    for word in gensim.utils.simple_preprocess(text):
        if word not in stopwords_list:
            word_list.append(word)
    final_list = ' '.join(word_list)
    return final_list


In [ ]:
combined_df['stopword_clean'] = combined_df['body'].apply(lambda x:stopword_clean(x))
print(combined_df['stopword_clean'].head(100))
combined_df.to_csv('combined_forum_cleaned.csv', index=False)

0     enter answer without spaces crazy trying kinds...
1                                             smartbook
2     answer question tried everything nothing seems...
3     passed output printed bubble line misleading u...
4     problem question try advise bogus variable ans...
                            ...                        
95    difference result returned string instead inte...
96    problem define result loop means result set ba...
97    reason beyond knowledge assign myfileread vari...
98                        problem set file input output
99                        problem set file input output
Name: stopword_clean, Length: 100, dtype: object


In [12]:
def remove_names_and_cities(text):
     doc = nlp(text)
     clean_text = text
     for ent in doc.ents:
            if ent.label_ == 'PERSON' or ent.label_ == 'GPE':
               clean_text = clean_text.replace(ent.text, '')
     return clean_text

In [13]:
# combined_df['remove_names_and_cities'] = combined_df['body'].apply(lambda x:remove_names_and_cities(x))
# print(combined_df.head(5))
# mini_df = combined_df.head(20)
# mini_df['remove_names_and_cities'] = mini_df['body'].apply(lambda x:remove_names_and_cities(x))
# print(mini_df['remove_names_and_cities'].head(20))
# print(combined_df['remove_names_and_cities'].head(100))

In [15]:
print(combined_df.columns)

Index(['id', 'course_id', 'body', 'stopword_clean'], dtype='object')


In [17]:
combined_df = combined_df.dropna(subset=['stopword_clean'])